# Structured Output

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to use `response_format` to get validated, typed responses from agents.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define a Pydantic Schema

Create a Pydantic `BaseModel` that describes the structure you want the LLM to return.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class WeatherReport(BaseModel):
    city: str = Field(description="The city name")
    temperature: float = Field(description="Temperature in Fahrenheit")
    conditions: str = Field(description="Weather conditions description")
    recommendation: str = Field(description="What to wear or bring")

print(WeatherReport.model_json_schema())

## 4. Create an Agent with response_format

Pass the schema to `response_format` to enforce structured output.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[],
    response_format=WeatherReport,
)

result = agent.invoke({
    "messages": [HumanMessage(content="Give me a weather report for San Francisco")]
})

response = result["structured_response"]
print(f"City: {response.city}")
print(f"Temperature: {response.temperature}°F")
print(f"Conditions: {response.conditions}")
print(f"Recommendation: {response.recommendation}")

## 5. ToolStrategy vs ProviderStrategy

`ToolStrategy` wraps the schema as a tool call. `ProviderStrategy` uses the provider's native API.

In [ ]:
from langgraph.prebuilt.chat_agent_executor import ToolStrategy, ProviderStrategy

agent_tool = create_react_agent(
    model=model,
    tools=[],
    response_format=(WeatherReport, ToolStrategy),
)

result = agent_tool.invoke({
    "messages": [HumanMessage(content="Weather report for Tokyo")]
})
print("ToolStrategy:", result["structured_response"])

In [ ]:
agent_provider = create_react_agent(
    model=model,
    tools=[],
    response_format=(WeatherReport, ProviderStrategy),
)

result = agent_provider.invoke({
    "messages": [HumanMessage(content="Weather report for London")]
})
print("ProviderStrategy:", result["structured_response"])

## 6. Other Schema Types

Besides Pydantic, you can use TypedDict and dataclass.

In [ ]:
from typing import TypedDict

class ExtractedInfo(TypedDict):
    name: str
    age: int
    hobbies: List[str]

agent_td = create_react_agent(
    model=model,
    tools=[],
    response_format=ExtractedInfo,
)

result = agent_td.invoke({
    "messages": [HumanMessage(content="Extract info: John is 30 and likes hiking and painting")]
})
print("TypedDict result:", result["structured_response"])

## 7. Structured Output with Tools

The agent can call tools to gather data, then format the final response as structured output.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup_product(name: str) -> dict:
    """Look up product details by name."""
    products = {
        "laptop": {"price": 999.99, "stock": 45, "category": "electronics"},
        "headphones": {"price": 79.99, "stock": 120, "category": "audio"},
    }
    return products.get(name.lower(), {"error": "Product not found"})

class ProductReport(BaseModel):
    product_name: str = Field(description="Name of the product")
    price: float = Field(description="Price in USD")
    in_stock: bool = Field(description="Whether the product is available")
    recommendation: str = Field(description="Purchase recommendation")

agent = create_react_agent(
    model=model,
    tools=[lookup_product],
    response_format=ProductReport,
)

result = agent.invoke({
    "messages": [HumanMessage(content="Tell me about the laptop")]
})

report = result["structured_response"]
print(f"{report.product_name}: ${report.price} - {report.recommendation}")

## Summary

- Use `response_format` on `create_react_agent` to enforce structured LLM output
- Pydantic `BaseModel` is the most common schema type with full validation
- Access parsed results via `result["structured_response"]`
- `ToolStrategy` works broadly; `ProviderStrategy` uses native provider APIs
- Supported types: Pydantic, dataclass, TypedDict, JSON Schema